# BeautifulSoup Extraction Patterns

This notebook explains the scraping logic behind the Day 2 scripts.

The core workflow is:

1. start with HTML text
2. parse it into a `soup` object
3. identify repeated record containers
4. select fields inside each record
5. extract visible text or attributes
6. store one row per record
7. convert rows into a DataFrame

This is the same logic used in the static scraping walkthrough, but here the HTML is small enough to inspect by eye.


## 1. Imports

We use two packages:

- `BeautifulSoup` from `bs4` parses HTML and lets us search it.
- `pandas` turns extracted rows into tables.


In [ ]:
# BeautifulSoup is the HTML parser/search tool.
from bs4 import BeautifulSoup

# pandas is used after extraction, when we want a rectangular table.
import pandas as pd


## 2. Example HTML

The artificial page below contains two repeated records. Each record is an `<article>` with:

- a title link: `<h2><a href=...>`
- a summary paragraph: `<p class="summary">`
- an author: `<span class="author">`
- tag links inside `<ul class="tags">`
- attributes such as `class` and `data-id`

This is a small version of the kind of structure you might see for posts, search results, videos, or product cards.


In [ ]:
# This is raw HTML stored as a Python string.
# In a real scraper, similar HTML usually comes from response.text.
html = """
<html>
  <body>
    <section id="results">
      <article class="result card promoted" data-id="p1">
        <h2><a href="/posts/p1">First post</a></h2>
        <p class="summary">This is the first summary.</p>
        <span class="author">Noelle</span>
        <ul class="tags">
          <li><a class="tag" href="/tags/api">api</a></li>
          <li><a class="tag" href="/tags/platforms">platforms</a></li>
        </ul>
      </article>

      <article class="result card" data-id="p2">
        <h2><a href="/posts/p2">Second post</a></h2>
        <p class="summary">This is the second summary.</p>
        <span class="author">Alex</span>
        <ul class="tags">
          <li><a class="tag" href="/tags/scraping">scraping</a></li>
        </ul>
      </article>
    </section>
  </body>
</html>
"""

print(type(html))
print(html[:300])


At this point, `html` is just text. Python does not yet understand the tags, links, classes, or attributes. BeautifulSoup creates a searchable representation of that HTML.


In [ ]:
# BeautifulSoup turns HTML text into a searchable parse tree.
# "html.parser" is Python's built-in HTML parser.
soup = BeautifulSoup(html, "html.parser")

# prettify() is for inspection: it prints the nested HTML with indentation.
print(soup.prettify()[:1000])


## 3. HTML Elements: Tag, Text, Attribute

An HTML element can contain:

- a tag name, such as `article`, `a`, `span`, or `p`
- visible text, such as `First post`
- attributes, such as `class`, `href`, or `data-id`

Example: `<a class="tag" href="/tags/api">api</a>`

- tag name: `a`
- visible text: `api`
- class attribute: `tag`
- href attribute: `/tags/api`


In [ ]:
# select_one("a") returns the first <a> element.
first_link = soup.select_one("a")

print("Whole element:", first_link)
print("Tag name:", first_link.name)

# get_text(strip=True) extracts visible text and removes surrounding whitespace.
print("Visible text:", first_link.get_text(strip=True))

# .get("href") extracts an attribute. If it is missing, .get() returns None.
print("href attribute:", first_link.get("href"))
print("class attribute:", first_link.get("class"))


## 4. What Selectors Are

A selector tells BeautifulSoup which HTML elements to find.

Common selector forms:

| Selector | Meaning |
|---|---|
| `article` | all `<article>` elements |
| `a` | all `<a>` link elements |
| `a[href]` | all links that have an `href` attribute |
| `.result` | all elements with class `result` |
| `.summary` | all elements with class `summary` |
| `#results` | the element with id `results` |
| `.tags a.tag` | tag links inside an element with class `tags` |

The dot in `.result` means class. The hash in `#results` means id.


## 5. `select_one()` vs `select()`

- `select_one(selector)` returns the first matching element, or `None`.
- `select(selector)` returns a list of all matching elements.

Use `select_one()` when you expect one value. Use `select()` when you expect several values.


In [ ]:
# First matching element with class="result".
first_card = soup.select_one(".result")
print(first_card.get("data-id"))

# All matching elements with class="result".
all_cards = soup.select(".result")
print(len(all_cards))


## 6. Multiple Classes: `.result.card` vs `.result .card`

No space: `.result.card`

Means: one element that has both classes `result` and `card`.

With a space: `.result .card`

Means: an element with class `card` somewhere inside an element with class `result`.

Those are different structural claims about the page.


In [ ]:
# No space: elements that have both classes at the same time.
for card in soup.select(".result.card"):
    print(card.get("data-id"), card.get("class"))


In [ ]:
# With a space: .card elements nested inside .result elements.
# Our example has no nested .card elements, so this returns an empty list.
print(soup.select(".result .card"))


## 7. Select Inside One Record

A central scraping rule:

First select the repeated record container, then select fields inside that container.

This keeps the title, summary, author, and tags from the same record together.


In [ ]:
# One repeated record container.
card = soup.select_one(".result.card")

# These selectors are searched inside this one card, not across the whole page.
title_link = card.select_one("h2 a")
summary = card.select_one(".summary")
author = card.select_one(".author")
tag_links = card.select(".tags a.tag")

print("title element:", title_link)
print("summary element:", summary)
print("author element:", author)
print("tag elements:", tag_links)


## 8. Extract Visible Text

`get_text(" ", strip=True)` is used often in the scripts.

It means:

- extract visible text
- join nested text with spaces
- remove leading and trailing whitespace

This matters because HTML often contains line breaks and indentation that should not become data.


In [ ]:
# Extract visible text from selected elements.
title_text = title_link.get_text(" ", strip=True)
summary_text = summary.get_text(" ", strip=True)
author_name = author.get_text(" ", strip=True)

print(title_text)
print(summary_text)
print(author_name)


## 9. Extract Attributes

Some useful data is not visible text.

For example:

- the post URL is in the `href` attribute
- the record ID is in the `data-id` attribute
- class labels are in the `class` attribute

Use `.get("attribute_name")` for attributes.


In [ ]:
# data-id is an attribute on the article element.
post_id = card.get("data-id")

# href is an attribute on the title link.
post_url = title_link.get("href")

# class is also an attribute; BeautifulSoup returns class labels as a list.
classes = card.get("class")

print(post_id)
print(post_url)
print(classes)


## 10. Defensive Extraction

Real pages are inconsistent. One card may be missing an author, date, image, or link.

This pattern prevents a crash and stores missingness explicitly:

`value = element.get_text(" ", strip=True) if element else None`


In [ ]:
# There is no element with class="date" in this example.
date_element = card.select_one(".date")
print(date_element)

# Safe extraction: store None if the element is missing.
date_text = date_element.get_text(" ", strip=True) if date_element else None
print(date_text)


## 11. One Record Becomes One Row

A row is usually a dictionary. The dictionary keys become column names.


In [ ]:
row = {
    "post_id": card.get("data-id"),
    "title": title_link.get_text(" ", strip=True) if title_link else None,
    "url": title_link.get("href") if title_link else None,
    "summary": summary.get_text(" ", strip=True) if summary else None,
    "author": author.get_text(" ", strip=True) if author else None,
    "tags": [tag.get_text(" ", strip=True) for tag in tag_links],
    "tag_count": len(tag_links),
}

row


## 12. Many Records Become a DataFrame

Now repeat the same extraction for every record container.


In [ ]:
rows = []

# .result.card selects the repeated record containers.
for card in soup.select(".result.card"):
    # Search inside the current card to keep fields from the same record together.
    title_link = card.select_one("h2 a")
    summary = card.select_one(".summary")
    author = card.select_one(".author")
    tag_links = card.select(".tags a.tag")

    row = {
        "post_id": card.get("data-id"),
        "title": title_link.get_text(" ", strip=True) if title_link else None,
        "url": title_link.get("href") if title_link else None,
        "summary": summary.get_text(" ", strip=True) if summary else None,
        "author": author.get_text(" ", strip=True) if author else None,
        "tags": [tag.get_text(" ", strip=True) for tag in tag_links],
        "tag_count": len(tag_links),
        # Derived Boolean field: True if the card has class="promoted".
        "is_promoted": "promoted" in card.get("class", []),
    }

    rows.append(row)

# pd.DataFrame turns the list of row dictionaries into a table.
df = pd.DataFrame(rows)
df


## 13. Long Table for Repeated Values

The `tags` field has multiple values. Often it is better to create a separate table with one row per post-tag pair.


In [ ]:
tag_rows = []

for card in soup.select(".result.card"):
    post_id = card.get("data-id")
    title_link = card.select_one("h2 a")
    title = title_link.get_text(" ", strip=True) if title_link else None

    for tag in card.select(".tags a.tag"):
        tag_rows.append(
            {
                "post_id": post_id,
                "title": title,
                "tag": tag.get_text(" ", strip=True),
                "tag_url": tag.get("href"),
            }
        )

tags_df = pd.DataFrame(tag_rows)
tags_df


## 14. Direct Children vs Descendants

A space means descendant at any depth: `#results a` means all links anywhere inside `#results`.

The `>` symbol means direct child: `#results > article` means articles immediately inside `#results`.


In [ ]:
# Descendant selector: all links anywhere inside #results.
all_links_inside_results = soup.select("#results a")
print([link.get_text(" ", strip=True) for link in all_links_inside_results])


In [ ]:
# Direct-child selector: only article elements immediately inside #results.
direct_articles = soup.select("#results > article")
print([article.get("data-id") for article in direct_articles])


## 15. How This Maps to the Static Scraping Walkthrough

In the static scraping walkthrough, the page is `https://quotes.toscrape.com/`.

There, the repeated container is `.quote`.

Inside each `.quote`, the script looks for:

- `.text`: quote text
- `.author`: author name
- `.tags a.tag`: tag links
- `span a[href]`: author profile link

The page is different, but the workflow is the same:

1. find repeated record containers
2. inspect one record
3. select fields inside one record
4. loop over records
5. build rows
6. create a DataFrame


## From Patterns to Workflow

This notebook focuses on selector mechanics: how to read HTML, choose selectors,
extract text and attributes, and turn repeated elements into rows.

The full scraping workflow, including real-page collection, raw HTML storage,
link extraction, pagination, and output files, is in
`day_2_static_scraping_walkthrough.ipynb`.


## 16. Mini Exercise

Using the example HTML in this notebook:

1. Explain what `.result.card` selects.
2. Explain why `.result .card` returns nothing here.
3. Add a column called `n_words_summary` that counts words in each summary.
4. Create a long table with one row per post-tag combination.
5. Explain which fields come from visible text and which fields come from attributes.

## Key Takeaway

BeautifulSoup does not decide what counts as a record or field. You decide that through selectors.

That means selectors are methodological choices, not just technical details.
